# 06 Router Training with Comprehensive W&B Tracking

**Train a multimodal VLM router with complete experiment tracking**

## What this notebook does:

1. **Dataset Loading**: Load final prepared datasets with hierarchical labels
2. **Model Architecture**: Vision + Text + Metadata → Transformer → Classification
3. **Training**: Full W&B tracking with real-time plots
4. **Evaluation**: Router accuracy, cost savings, regret analysis
5. **Baselines**: Compare against heuristic and single-model strategies
6. **Visualization**: Training curves, confusion matrices, routing patterns

## Key Features:
- Complete W&B integration for experiment tracking
- Real-time training plots and metrics
- Comprehensive evaluation with baselines
- Error analysis and debugging tools
- Model checkpointing and resuming

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Imports & Setup

In [ ]:
import os
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, asdict
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR

import wandb
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Transformers for vision encoder and tokenizer
from transformers import (
    AutoModel,
    AutoTokenizer,
    AutoImageProcessor,
    CLIPVisionModel,
    CLIPImageProcessor,
)

# Local imports
from imports.common_utils import return_model_specs, return_model_pricing

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"MPS available: {torch.backends.mps.is_available()}")

## 2. Configuration

In [ ]:
@dataclass
class RouterConfig:
    """Complete configuration for router training."""
    
    # Paths
    data_root: Path = Path.cwd().parent.parent.parent / "dataset" / "final_dataset"
    image_root: Path = Path.cwd().parent.parent.parent / "dataset" / "which_vlm_data" / "images"
    output_dir: Path = Path.cwd() / "router_training_outputs"
    checkpoint_dir: Path = Path.cwd() / "router_checkpoints"
    
    # Model architecture
    vision_encoder_name: str = "openai/clip-vit-base-patch32"  # Small, fast CLIP
    text_tokenizer_name: str = "bert-base-uncased"
    d_model: int = 384  # Hidden dimension
    num_layers: int = 4  # Transformer layers
    num_heads: int = 6   # Attention heads
    ffn_dim: int = 1536  # FFN dimension (4 * d_model)
    dropout: float = 0.1
    max_text_length: int = 256  # Max tokens for text input
    
    # Training
    batch_size: int = 32
    num_epochs: int = 20
    learning_rate: float = 1e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    gradient_clip_norm: float = 1.0
    
    # Loss configuration
    loss_type: str = "combined"  # "ce", "kl", or "combined"
    ce_weight: float = 0.5  # Weight for CE loss in combined mode
    kl_weight: float = 0.5  # Weight for KL loss in combined mode
    label_smoothing: float = 0.0  # Label smoothing for CE
    
    # Data
    num_workers: int = 4
    pin_memory: bool = True
    
    # Evaluation
    eval_every_n_steps: int = 500
    save_every_n_steps: int = 1000
    
    # W&B
    wandb_project: str = "vlm-router-training"
    wandb_entity: Optional[str] = None  # Set to your W&B username/team
    wandb_run_name: Optional[str] = None
    log_every_n_steps: int = 10
    
    # Device
    device: str = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
    
    # Random seed
    seed: int = 42
    
    def __post_init__(self):
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Create config
config = RouterConfig()

# Set random seeds
torch.manual_seed(config.seed)
np.random.seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)

print("\n=== Configuration ===")
for key, value in asdict(config).items():
    print(f"{key:25s}: {value}")

In [ ]:
# Load model specs
MODEL_SPECS = return_model_specs()
MODEL_PRICING = return_model_pricing()

model_names = [m['name'] for m in MODEL_SPECS]
ID_TO_NAME = {m['id']: m['name'] for m in MODEL_SPECS}
NAME_TO_ID = {m['name']: m['id'] for m in MODEL_SPECS}
NUM_MODELS = len(MODEL_SPECS)

print(f"\nNumber of models to route between: {NUM_MODELS}")

## 3. Load Dataset

In [ ]:
# Load prepared datasets
print("Loading prepared router datasets...")

train_df = pd.read_parquet(config.data_root / "router_final" / "router_train_final.parquet")
val_df = pd.read_parquet(config.data_root / "router_final" / "router_val_final.parquet")
test_df = pd.read_parquet(config.data_root / "router_final" / "router_test_final.parquet")

print(f"\nDataset sizes:")
print(f"  Train: {len(train_df):,} samples")
print(f"  Val:   {len(val_df):,} samples")
print(f"  Test:  {len(test_df):,} samples")

print(f"\nColumns in dataset: {train_df.columns.tolist()[:20]}...")  # Show first 20

In [ ]:
# Inspect label distribution
print("\n=== Label Distribution (Training Set) ===")
label_dist = train_df['router_best_model_name'].value_counts()
print(label_dist)
print("\nFractions:")
print(label_dist / len(train_df))

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
label_dist.plot(kind='bar', ax=ax)
ax.set_title('Training Label Distribution')
ax.set_xlabel('Model')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Dataset Class

In [ ]:
class RouterDataset(Dataset):
    """Dataset for router training."""
    
    def __init__(
        self,
        df: pd.DataFrame,
        image_root: Path,
        image_processor,
        tokenizer,
        max_text_length: int = 256,
        model_names: List[str] = None,
        use_soft_labels: bool = True,
    ):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.image_processor = image_processor
        self.tokenizer = tokenizer
        self.max_text_length = max_text_length
        self.model_names = model_names or model_names
        self.use_soft_labels = use_soft_labels
        
    def __len__(self):
        return len(self.df)
    
    def _construct_router_text(self, row) -> str:
        """Construct the text input for the router."""
        parts = []
        
        # Task information
        if pd.notna(row.get('router_task')):
            parts.append(f"Task: {row['router_task']}")
        
        # Dataset information
        if pd.notna(row.get('source_dataset')):
            parts.append(f"Dataset: {row['source_dataset']}")
        
        # Question type
        if pd.notna(row.get('txt_question_type')):
            parts.append(f"QType: {row['txt_question_type']}")
        
        # Has multiple choice
        if pd.notna(row.get('txt_has_mc_options')):
            has_mc = "Yes" if row['txt_has_mc_options'] else "No"
            parts.append(f"HasMC: {has_mc}")
        
        # Image dimensions
        if pd.notna(row.get('img_width')) and pd.notna(row.get('img_height')):
            parts.append(f"ImgSize: {int(row['img_width'])}x{int(row['img_height'])}")
        
        # Aspect ratio
        if pd.notna(row.get('img_aspect_ratio')):
            parts.append(f"AR: {row['img_aspect_ratio']:.2f}")
        
        # Prompt length
        if pd.notna(row.get('txt_prompt_length_words')):
            parts.append(f"PromptWords: {int(row['txt_prompt_length_words'])}")
        
        # Add the actual prompt
        if pd.notna(row.get('prompt_raw')):
            parts.append(f"Prompt: {row['prompt_raw']}")
        
        return " | ".join(parts)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load and process image
        image_path = row.get('image_path')
        if pd.isna(image_path) or not Path(image_path).exists():
            # Create a blank placeholder image if missing
            image = Image.new('RGB', (224, 224), color='gray')
        else:
            try:
                image = Image.open(image_path).convert('RGB')
            except Exception as e:
                print(f"Error loading image {image_path}: {e}")
                image = Image.new('RGB', (224, 224), color='gray')
        
        # Process image
        pixel_values = self.image_processor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)
        
        # Construct and tokenize text
        router_text = self._construct_router_text(row)
        text_encoding = self.tokenizer(
            router_text,
            max_length=self.max_text_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        
        # Hard label
        hard_label = row['router_best_model_id']
        
        # Soft labels (if available)
        soft_labels = torch.zeros(len(self.model_names), dtype=torch.float32)
        if self.use_soft_labels:
            for i, model_name in enumerate(self.model_names):
                col = f'router_soft_p_{model_name}'
                if col in row.index and pd.notna(row[col]):
                    soft_labels[i] = row[col]
            # Normalize to ensure sum=1
            if soft_labels.sum() > 0:
                soft_labels = soft_labels / soft_labels.sum()
            else:
                # Fallback to one-hot if soft labels missing
                soft_labels[hard_label] = 1.0
        else:
            soft_labels[hard_label] = 1.0
        
        return {
            'pixel_values': pixel_values,
            'input_ids': text_encoding['input_ids'].squeeze(0),
            'attention_mask': text_encoding['attention_mask'].squeeze(0),
            'hard_label': torch.tensor(hard_label, dtype=torch.long),
            'soft_labels': soft_labels,
            'sample_id': row.get('sample_id', f'sample_{idx}'),
        }

print("RouterDataset class defined.")

## 5. Model Architecture

In [ ]:
class MultimodalRouterModel(nn.Module):
    """Multimodal router: Vision + Text → Model selection."""
    
    def __init__(
        self,
        vision_encoder_name: str,
        text_tokenizer_name: str,
        num_models: int,
        d_model: int = 384,
        num_layers: int = 4,
        num_heads: int = 6,
        ffn_dim: int = 1536,
        dropout: float = 0.1,
        freeze_vision: bool = True,
    ):
        super().__init__()
        
        self.d_model = d_model
        self.num_models = num_models
        
        # Vision encoder (frozen CLIP)
        print(f"Loading vision encoder: {vision_encoder_name}")
        self.vision_encoder = CLIPVisionModel.from_pretrained(vision_encoder_name)
        vision_dim = self.vision_encoder.config.hidden_size
        
        if freeze_vision:
            for param in self.vision_encoder.parameters():
                param.requires_grad = False
            print("Vision encoder frozen.")
        
        # Vision projection to d_model
        self.vision_proj = nn.Linear(vision_dim, d_model)
        
        # Text embedding
        print(f"Loading tokenizer: {text_tokenizer_name}")
        tokenizer = AutoTokenizer.from_pretrained(text_tokenizer_name)
        vocab_size = len(tokenizer)
        self.text_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(512, d_model)  # Max 512 positions
        
        # CLS token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_models),
        )
        
        print(f"\nRouter model initialized:")
        print(f"  Vision dim: {vision_dim} → {d_model}")
        print(f"  Text vocab: {vocab_size} → {d_model}")
        print(f"  Transformer: {num_layers} layers, {num_heads} heads")
        print(f"  Output: {num_models} models")
    
    def forward(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            pixel_values: [B, 3, H, W]
            input_ids: [B, T]
            attention_mask: [B, T]
        
        Returns:
            logits: [B, num_models]
        """
        batch_size = pixel_values.size(0)
        
        # Vision encoding
        with torch.no_grad() if not self.vision_encoder.training else torch.enable_grad():
            vision_outputs = self.vision_encoder(pixel_values)
            vision_features = vision_outputs.pooler_output  # [B, vision_dim]
        
        vision_token = self.vision_proj(vision_features).unsqueeze(1)  # [B, 1, d_model]
        
        # Text encoding
        text_emb = self.text_embedding(input_ids)  # [B, T, d_model]
        
        # Add positional embeddings
        seq_length = text_emb.size(1)
        positions = torch.arange(seq_length, device=text_emb.device).unsqueeze(0).expand(batch_size, -1)
        text_emb = text_emb + self.position_embedding(positions)
        
        # Build multimodal sequence: [CLS, vision, text]
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)  # [B, 1, d_model]
        sequence = torch.cat([cls_tokens, vision_token, text_emb], dim=1)  # [B, 1+1+T, d_model]
        
        # Build attention mask
        # CLS and vision are always attended
        cls_mask = torch.ones(batch_size, 1, device=attention_mask.device)
        vision_mask = torch.ones(batch_size, 1, device=attention_mask.device)
        full_mask = torch.cat([cls_mask, vision_mask, attention_mask], dim=1)  # [B, 2+T]
        
        # Transformer expects inverted mask (0 = attend, 1 = ignore) for some versions
        # But batch_first TransformerEncoder uses standard mask (1 = attend)
        # Create src_key_padding_mask: True = ignore
        src_key_padding_mask = (full_mask == 0)
        
        # Transformer encoding
        hidden = self.transformer(sequence, src_key_padding_mask=src_key_padding_mask)  # [B, L, d_model]
        
        # Use CLS token for classification
        cls_hidden = hidden[:, 0, :]  # [B, d_model]
        
        # Classification
        logits = self.classifier(cls_hidden)  # [B, num_models]
        
        return logits

print("MultimodalRouterModel class defined.")

## 6. Initialize Model & Data Loaders

In [ ]:
# Load image processor and tokenizer
print("Loading image processor and tokenizer...")
image_processor = CLIPImageProcessor.from_pretrained(config.vision_encoder_name)
tokenizer = AutoTokenizer.from_pretrained(config.text_tokenizer_name)

# Create datasets
print("\nCreating datasets...")
train_dataset = RouterDataset(
    train_df,
    config.image_root,
    image_processor,
    tokenizer,
    config.max_text_length,
    model_names,
    use_soft_labels=True,
)

val_dataset = RouterDataset(
    val_df,
    config.image_root,
    image_processor,
    tokenizer,
    config.max_text_length,
    model_names,
    use_soft_labels=True,
)

test_dataset = RouterDataset(
    test_df,
    config.image_root,
    image_processor,
    tokenizer,
    config.max_text_length,
    model_names,
    use_soft_labels=True,
)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset:   {len(val_dataset)} samples")
print(f"Test dataset:  {len(test_dataset)} samples")

In [ ]:
# Create data loaders
print("\nCreating data loaders...")
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=config.pin_memory,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size * 2,  # Larger batch for eval
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=config.pin_memory,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.batch_size * 2,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=config.pin_memory,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
# Initialize model
print("\n=== Initializing Router Model ===")
model = MultimodalRouterModel(
    vision_encoder_name=config.vision_encoder_name,
    text_tokenizer_name=config.text_tokenizer_name,
    num_models=NUM_MODELS,
    d_model=config.d_model,
    num_layers=config.num_layers,
    num_heads=config.num_heads,
    ffn_dim=config.ffn_dim,
    dropout=config.dropout,
    freeze_vision=True,
)

model = model.to(config.device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

## 7. Training Setup

In [ ]:
# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)

# Learning rate scheduler
total_steps = len(train_loader) * config.num_epochs
warmup_steps = int(total_steps * config.warmup_ratio)

scheduler = OneCycleLR(
    optimizer,
    max_lr=config.learning_rate,
    total_steps=total_steps,
    pct_start=config.warmup_ratio,
    anneal_strategy='cos',
)

print(f"\nOptimizer: AdamW (lr={config.learning_rate}, wd={config.weight_decay})")
print(f"Scheduler: OneCycleLR (total_steps={total_steps}, warmup={warmup_steps})")

In [ ]:
# Loss functions
def compute_loss(
    logits: torch.Tensor,
    hard_labels: torch.Tensor,
    soft_labels: torch.Tensor,
    loss_type: str = "combined",
    ce_weight: float = 0.5,
    kl_weight: float = 0.5,
    label_smoothing: float = 0.0,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """
    Compute loss based on configuration.
    
    Returns:
        loss: Total loss
        metrics: Dict with loss components
    """
    metrics = {}
    
    if loss_type == "ce":
        # Cross-entropy with hard labels
        ce_loss = F.cross_entropy(logits, hard_labels, label_smoothing=label_smoothing)
        loss = ce_loss
        metrics['ce_loss'] = ce_loss.item()
        
    elif loss_type == "kl":
        # KL divergence with soft labels
        log_probs = F.log_softmax(logits, dim=-1)
        kl_loss = F.kl_div(log_probs, soft_labels, reduction='batchmean')
        loss = kl_loss
        metrics['kl_loss'] = kl_loss.item()
        
    elif loss_type == "combined":
        # Combined CE + KL
        ce_loss = F.cross_entropy(logits, hard_labels, label_smoothing=label_smoothing)
        log_probs = F.log_softmax(logits, dim=-1)
        kl_loss = F.kl_div(log_probs, soft_labels, reduction='batchmean')
        loss = ce_weight * ce_loss + kl_weight * kl_loss
        metrics['ce_loss'] = ce_loss.item()
        metrics['kl_loss'] = kl_loss.item()
        metrics['combined_loss'] = loss.item()
    
    else:
        raise ValueError(f"Unknown loss_type: {loss_type}")
    
    metrics['total_loss'] = loss.item()
    return loss, metrics

print("Loss function defined.")

## 8. Initialize W&B

In [ ]:
# Initialize W&B
wandb.init(
    project=config.wandb_project,
    entity=config.wandb_entity,
    name=config.wandb_run_name,
    config=asdict(config),
    tags=['router', 'multimodal', 'vlm'],
)

# Log model architecture
wandb.watch(model, log='all', log_freq=100)

print(f"\nW&B run initialized: {wandb.run.name}")
print(f"W&B run URL: {wandb.run.url}")

## 9. Training & Evaluation Functions

In [ ]:
def train_epoch(
    model: nn.Module,
    train_loader: DataLoader,
    optimizer,
    scheduler,
    epoch: int,
    config: RouterConfig,
) -> Dict[str, float]:
    """Train for one epoch."""
    model.train()
    
    total_loss = 0.0
    total_ce_loss = 0.0
    total_kl_loss = 0.0
    correct = 0
    total = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}")
    
    for step, batch in enumerate(progress_bar):
        # Move to device
        pixel_values = batch['pixel_values'].to(config.device)
        input_ids = batch['input_ids'].to(config.device)
        attention_mask = batch['attention_mask'].to(config.device)
        hard_labels = batch['hard_label'].to(config.device)
        soft_labels = batch['soft_labels'].to(config.device)
        
        # Forward pass
        logits = model(pixel_values, input_ids, attention_mask)
        
        # Compute loss
        loss, loss_metrics = compute_loss(
            logits,
            hard_labels,
            soft_labels,
            loss_type=config.loss_type,
            ce_weight=config.ce_weight,
            kl_weight=config.kl_weight,
            label_smoothing=config.label_smoothing,
        )
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip_norm)
        
        optimizer.step()
        scheduler.step()
        
        # Metrics
        total_loss += loss_metrics['total_loss']
        if 'ce_loss' in loss_metrics:
            total_ce_loss += loss_metrics['ce_loss']
        if 'kl_loss' in loss_metrics:
            total_kl_loss += loss_metrics['kl_loss']
        
        preds = logits.argmax(dim=-1)
        correct += (preds == hard_labels).sum().item()
        total += hard_labels.size(0)
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': f"{loss_metrics['total_loss']:.4f}",
            'acc': f"{correct / total:.4f}",
            'lr': f"{scheduler.get_last_lr()[0]:.2e}",
        })
        
        # Log to W&B
        global_step = epoch * len(train_loader) + step
        if step % config.log_every_n_steps == 0:
            log_dict = {
                'train/loss': loss_metrics['total_loss'],
                'train/accuracy': correct / total,
                'train/learning_rate': scheduler.get_last_lr()[0],
                'train/epoch': epoch,
                'train/step': global_step,
            }
            if 'ce_loss' in loss_metrics:
                log_dict['train/ce_loss'] = loss_metrics['ce_loss']
            if 'kl_loss' in loss_metrics:
                log_dict['train/kl_loss'] = loss_metrics['kl_loss']
            
            wandb.log(log_dict, step=global_step)
    
    # Epoch metrics
    metrics = {
        'loss': total_loss / len(train_loader),
        'accuracy': correct / total,
    }
    if config.loss_type in ['ce', 'combined']:
        metrics['ce_loss'] = total_ce_loss / len(train_loader)
    if config.loss_type in ['kl', 'combined']:
        metrics['kl_loss'] = total_kl_loss / len(train_loader)
    
    return metrics

print("train_epoch function defined.")

In [ ]:
@torch.no_grad()
def evaluate(
    model: nn.Module,
    data_loader: DataLoader,
    config: RouterConfig,
    split_name: str = 'val',
) -> Dict[str, float]:
    """Evaluate the model."""
    model.eval()
    
    total_loss = 0.0
    total_ce_loss = 0.0
    total_kl_loss = 0.0
    correct = 0
    total = 0
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    progress_bar = tqdm(data_loader, desc=f"Evaluating {split_name}")
    
    for batch in progress_bar:
        # Move to device
        pixel_values = batch['pixel_values'].to(config.device)
        input_ids = batch['input_ids'].to(config.device)
        attention_mask = batch['attention_mask'].to(config.device)
        hard_labels = batch['hard_label'].to(config.device)
        soft_labels = batch['soft_labels'].to(config.device)
        
        # Forward pass
        logits = model(pixel_values, input_ids, attention_mask)
        
        # Compute loss
        loss, loss_metrics = compute_loss(
            logits,
            hard_labels,
            soft_labels,
            loss_type=config.loss_type,
            ce_weight=config.ce_weight,
            kl_weight=config.kl_weight,
            label_smoothing=0.0,  # No smoothing during eval
        )
        
        total_loss += loss_metrics['total_loss']
        if 'ce_loss' in loss_metrics:
            total_ce_loss += loss_metrics['ce_loss']
        if 'kl_loss' in loss_metrics:
            total_kl_loss += loss_metrics['kl_loss']
        
        # Predictions
        probs = F.softmax(logits, dim=-1)
        preds = logits.argmax(dim=-1)
        correct += (preds == hard_labels).sum().item()
        total += hard_labels.size(0)
        
        # Store for metrics
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(hard_labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    
    # Compute metrics
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    metrics = {
        'loss': total_loss / len(data_loader),
        'accuracy': correct / total,
        'top1_accuracy': (all_preds == all_labels).mean(),
    }
    
    # Top-k accuracy
    for k in [2, 3]:
        if k <= NUM_MODELS:
            top_k_preds = np.argsort(all_probs, axis=1)[:, -k:]
            top_k_acc = np.array([label in preds for label, preds in zip(all_labels, top_k_preds)]).mean()
            metrics[f'top{k}_accuracy'] = top_k_acc
    
    if config.loss_type in ['ce', 'combined']:
        metrics['ce_loss'] = total_ce_loss / len(data_loader)
    if config.loss_type in ['kl', 'combined']:
        metrics['kl_loss'] = total_kl_loss / len(data_loader)
    
    return metrics, all_preds, all_labels, all_probs

print("evaluate function defined.")

## 10. Main Training Loop

In [ ]:
# Training loop
best_val_accuracy = 0.0
patience = 5
patience_counter = 0

print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60 + "\n")

for epoch in range(1, config.num_epochs + 1):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch}/{config.num_epochs}")
    print(f"{'='*60}\n")
    
    # Train
    train_metrics = train_epoch(model, train_loader, optimizer, scheduler, epoch, config)
    
    print(f"\nTrain metrics:")
    for k, v in train_metrics.items():
        print(f"  {k}: {v:.4f}")
    
    # Evaluate on validation set
    val_metrics, val_preds, val_labels, val_probs = evaluate(model, val_loader, config, 'val')
    
    print(f"\nValidation metrics:")
    for k, v in val_metrics.items():
        print(f"  {k}: {v:.4f}")
    
    # Log to W&B
    wandb_log = {
        'epoch': epoch,
        'train/epoch_loss': train_metrics['loss'],
        'train/epoch_accuracy': train_metrics['accuracy'],
        'val/loss': val_metrics['loss'],
        'val/accuracy': val_metrics['accuracy'],
        'val/top1_accuracy': val_metrics['top1_accuracy'],
    }
    
    if 'top2_accuracy' in val_metrics:
        wandb_log['val/top2_accuracy'] = val_metrics['top2_accuracy']
    if 'top3_accuracy' in val_metrics:
        wandb_log['val/top3_accuracy'] = val_metrics['top3_accuracy']
    
    if 'ce_loss' in train_metrics:
        wandb_log['train/epoch_ce_loss'] = train_metrics['ce_loss']
        wandb_log['val/ce_loss'] = val_metrics['ce_loss']
    if 'kl_loss' in train_metrics:
        wandb_log['train/epoch_kl_loss'] = train_metrics['kl_loss']
        wandb_log['val/kl_loss'] = val_metrics['kl_loss']
    
    wandb.log(wandb_log, step=epoch * len(train_loader))
    
    # Confusion matrix every few epochs
    if epoch % 5 == 0 or epoch == 1:
        cm = confusion_matrix(val_labels, val_preds)
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=model_names, yticklabels=model_names)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.set_title(f'Confusion Matrix (Epoch {epoch})')
        plt.tight_layout()
        wandb.log({f'val/confusion_matrix_epoch_{epoch}': wandb.Image(fig)}, step=epoch * len(train_loader))
        plt.close(fig)
    
    # Save best model
    if val_metrics['accuracy'] > best_val_accuracy:
        best_val_accuracy = val_metrics['accuracy']
        patience_counter = 0
        
        # Save checkpoint
        checkpoint_path = config.checkpoint_dir / 'best_model.pt'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_accuracy': val_metrics['accuracy'],
            'config': asdict(config),
        }, checkpoint_path)
        
        print(f"\n✓ New best model saved! Val accuracy: {best_val_accuracy:.4f}")
        
        # Log to W&B
        wandb.run.summary['best_val_accuracy'] = best_val_accuracy
        wandb.run.summary['best_epoch'] = epoch
        
        # Save as artifact
        artifact = wandb.Artifact(
            name=f'router-model-{wandb.run.id}',
            type='model',
            description=f'Best router model at epoch {epoch}',
        )
        artifact.add_file(str(checkpoint_path))
        wandb.log_artifact(artifact)
    else:
        patience_counter += 1
        print(f"\nNo improvement. Patience: {patience_counter}/{patience}")
    
    # Early stopping
    if patience_counter >= patience:
        print(f"\nEarly stopping triggered after {epoch} epochs.")
        break
    
    # Save periodic checkpoint
    if epoch % 5 == 0:
        checkpoint_path = config.checkpoint_dir / f'checkpoint_epoch_{epoch}.pt'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_accuracy': val_metrics['accuracy'],
            'config': asdict(config),
        }, checkpoint_path)
        print(f"\nCheckpoint saved: {checkpoint_path}")

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"\nBest validation accuracy: {best_val_accuracy:.4f}")

## 11. Final Evaluation on Test Set

In [ ]:
# Load best model
print("\nLoading best model for test evaluation...")
checkpoint = torch.load(config.checkpoint_dir / 'best_model.pt')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded model from epoch {checkpoint['epoch']} with val accuracy {checkpoint['val_accuracy']:.4f}")

# Evaluate on test set
test_metrics, test_preds, test_labels, test_probs = evaluate(model, test_loader, config, 'test')

print("\n" + "="*60)
print("TEST SET RESULTS")
print("="*60 + "\n")

for k, v in test_metrics.items():
    print(f"{k:20s}: {v:.4f}")
    wandb.run.summary[f'test/{k}'] = v

In [ ]:
# Test confusion matrix
cm = confusion_matrix(test_labels, test_preds)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=model_names, yticklabels=model_names)
ax.set_xlabel('Predicted Model', fontsize=12)
ax.set_ylabel('True Model', fontsize=12)
ax.set_title('Test Set Confusion Matrix', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

wandb.log({'test/confusion_matrix': wandb.Image(fig)})
plt.savefig(config.output_dir / 'test_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Per-class metrics
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    test_labels, test_preds, labels=range(NUM_MODELS)
)

per_class_df = pd.DataFrame({
    'model': model_names,
    'precision': precision,
    'recall': recall,
    'f1': f1,
    'support': support,
})

print("\n=== Per-Class Metrics (Test Set) ===")
print(per_class_df.to_string(index=False))

# Log as table
wandb.log({'test/per_class_metrics': wandb.Table(dataframe=per_class_df)})

## 12. End-to-End Cost/Performance Analysis

In [ ]:
# Simulate router performance using logged results
print("\n=== End-to-End Performance Analysis ===")
print("Simulating router routing decisions on test set...\n")

# Add predictions to test dataframe
test_df_eval = test_df.copy()
test_df_eval['router_predicted_model_id'] = test_preds
test_df_eval['router_predicted_model_name'] = [ID_TO_NAME[i] for i in test_preds]

# For each prediction, get the actual performance and cost
router_perfs = []
router_costs = []

for idx, row in test_df_eval.iterrows():
    pred_model_name = row['router_predicted_model_name']
    
    # Get performance of predicted model
    perf_col = f"{pred_model_name}__sample_score"
    cost_col = f"{pred_model_name}__cost"
    
    if perf_col in row.index and pd.notna(row[perf_col]):
        router_perfs.append(row[perf_col])
    else:
        router_perfs.append(0.0)
    
    if cost_col in row.index and pd.notna(row[cost_col]):
        router_costs.append(row[cost_col])
    else:
        router_costs.append(0.0)

test_df_eval['router_actual_perf'] = router_perfs
test_df_eval['router_actual_cost'] = router_costs

# Compute aggregate metrics
router_avg_perf = test_df_eval['router_actual_perf'].mean()
router_avg_cost = test_df_eval['router_actual_cost'].mean()

print(f"Router average performance: {router_avg_perf:.4f}")
print(f"Router average cost:        ${router_avg_cost:.6f}")

wandb.run.summary['test/router_avg_performance'] = router_avg_perf
wandb.run.summary['test/router_avg_cost'] = router_avg_cost

In [ ]:
# Compare to baselines
print("\n=== Baseline Comparisons ===")

# Baseline 1: Always use best model (oracle)
oracle_perf = test_df_eval['router_chosen_perf'].mean()
oracle_cost = test_df_eval['router_chosen_cost'].mean()
print(f"\nOracle (best per sample):")
print(f"  Avg performance: {oracle_perf:.4f}")
print(f"  Avg cost:        ${oracle_cost:.6f}")

# Baseline 2: Always use each individual model
baseline_results = []

for model_name in model_names:
    perf_col = f"{model_name}__sample_score"
    cost_col = f"{model_name}__cost"
    valid_col = f"{model_name}__valid_mask"
    
    # Filter to valid samples for this model
    valid_mask = test_df_eval[valid_col] if valid_col in test_df_eval.columns else [True] * len(test_df_eval)
    
    avg_perf = test_df_eval.loc[valid_mask, perf_col].mean() if perf_col in test_df_eval.columns else 0.0
    avg_cost = test_df_eval.loc[valid_mask, cost_col].mean() if cost_col in test_df_eval.columns else 0.0
    
    baseline_results.append({
        'strategy': f'Always {model_name}',
        'performance': avg_perf,
        'cost': avg_cost,
    })
    
    print(f"\nAlways {model_name}:")
    print(f"  Avg performance: {avg_perf:.4f}")
    print(f"  Avg cost:        ${avg_cost:.6f}")

# Add router and oracle to comparison
baseline_results.append({
    'strategy': 'Router (ours)',
    'performance': router_avg_perf,
    'cost': router_avg_cost,
})

baseline_results.append({
    'strategy': 'Oracle (best per sample)',
    'performance': oracle_perf,
    'cost': oracle_cost,
})

baseline_df = pd.DataFrame(baseline_results)
print("\n=== Summary Table ===")
print(baseline_df.to_string(index=False))

wandb.log({'test/baseline_comparison': wandb.Table(dataframe=baseline_df)})

In [ ]:
# Pareto plot: Cost vs Performance
fig, ax = plt.subplots(figsize=(12, 8))

# Plot baselines
for i, row in baseline_df.iterrows():
    if 'Router' in row['strategy']:
        marker = 'o'
        color = 'red'
        size = 200
        zorder = 10
    elif 'Oracle' in row['strategy']:
        marker = '*'
        color = 'gold'
        size = 300
        zorder = 9
    else:
        marker = 's'
        color = 'blue'
        size = 100
        zorder = 5
    
    ax.scatter(row['cost'], row['performance'], 
               marker=marker, s=size, color=color, alpha=0.7, zorder=zorder,
               label=row['strategy'], edgecolors='black', linewidths=1.5)

ax.set_xlabel('Average Cost (USD)', fontsize=12)
ax.set_ylabel('Average Performance', fontsize=12)
ax.set_title('Cost vs Performance: Router vs Baselines', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')
plt.tight_layout()

wandb.log({'test/cost_vs_performance': wandb.Image(fig)})
plt.savefig(config.output_dir / 'cost_vs_performance.png', dpi=150)
plt.show()

## 13. Routing Pattern Analysis

In [ ]:
# Analyze routing patterns by task
print("\n=== Routing Patterns by Task ===")

task_routing = (
    test_df_eval
    .groupby('router_task')['router_predicted_model_name']
    .value_counts(normalize=True)
    .rename('fraction')
    .reset_index()
)

# Pivot for visualization
task_routing_pivot = task_routing.pivot(
    index='router_task',
    columns='router_predicted_model_name',
    values='fraction'
).fillna(0)

# Plot
fig, ax = plt.subplots(figsize=(14, 8))
task_routing_pivot.plot(kind='bar', stacked=True, ax=ax, colormap='tab10')
ax.set_xlabel('Router Task', fontsize=12)
ax.set_ylabel('Fraction of Samples', fontsize=12)
ax.set_title('Router Model Selection by Task (Test Set)', fontsize=14, fontweight='bold')
ax.legend(title='Predicted Model', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

wandb.log({'test/routing_by_task': wandb.Image(fig)})
plt.savefig(config.output_dir / 'routing_by_task.png', dpi=150)
plt.show()

# Log as table
wandb.log({'test/routing_by_task_table': wandb.Table(dataframe=task_routing)})

In [ ]:
# Routing distribution overall
routing_dist = test_df_eval['router_predicted_model_name'].value_counts()

print("\n=== Overall Routing Distribution (Test Set) ===")
print(routing_dist)
print("\nFractions:")
print(routing_dist / len(test_df_eval))

fig, ax = plt.subplots(figsize=(10, 6))
routing_dist.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Router Predictions Distribution (Test Set)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

wandb.log({'test/routing_distribution': wandb.Image(fig)})
plt.savefig(config.output_dir / 'routing_distribution.png', dpi=150)
plt.show()

## 14. Save Final Results & Finish

In [ ]:
# Save test predictions
test_predictions_path = config.output_dir / 'test_predictions.parquet'
test_df_eval[[
    'sample_id',
    'router_task',
    'source_dataset',
    'router_best_model_name',  # Oracle choice
    'router_predicted_model_name',  # Router choice
    'router_chosen_perf',  # Oracle performance
    'router_actual_perf',  # Router performance
    'router_chosen_cost',  # Oracle cost
    'router_actual_cost',  # Router cost
]].to_parquet(test_predictions_path, index=False)

print(f"\nTest predictions saved to: {test_predictions_path}")

# Upload as artifact
artifact = wandb.Artifact(
    name=f'test-predictions-{wandb.run.id}',
    type='predictions',
    description='Test set predictions with performance and cost',
)
artifact.add_file(str(test_predictions_path))
wandb.log_artifact(artifact)

In [ ]:
# Final summary
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"\nBest validation accuracy:     {best_val_accuracy:.4f}")
print(f"Test accuracy (top-1):        {test_metrics['accuracy']:.4f}")
if 'top2_accuracy' in test_metrics:
    print(f"Test accuracy (top-2):        {test_metrics['top2_accuracy']:.4f}")
if 'top3_accuracy' in test_metrics:
    print(f"Test accuracy (top-3):        {test_metrics['top3_accuracy']:.4f}")
print(f"\nRouter avg performance:       {router_avg_perf:.4f}")
print(f"Router avg cost:              ${router_avg_cost:.6f}")
print(f"\nOracle avg performance:       {oracle_perf:.4f}")
print(f"Oracle avg cost:              ${oracle_cost:.6f}")
print(f"\nPerformance gap to oracle:    {(oracle_perf - router_avg_perf):.4f}")
print(f"Cost savings vs oracle:       ${(oracle_cost - router_avg_cost):.6f}")

print(f"\nCheckpoints saved to: {config.checkpoint_dir}")
print(f"Outputs saved to: {config.output_dir}")
print(f"W&B run: {wandb.run.url}")

print("\n" + "="*60)
print("ALL DONE!")
print("="*60)

In [ ]:
# Finish W&B run
wandb.finish()
print("\nW&B run finished.")